# Authors
Google (thanks Baruch!) + Nathan + Gabrael, minor modifications from Ankush and JC for CS 123 purposes

# GPU
Please connect to an **A100** GPU to run the following notebook.

# Log in to Weights and Biases

Paste your wandb key in the section below to log all your training processes

In [ ]:
from google.colab import userdata

wandb_key = ''

In [4]:
!pip install -q wandb
import wandb
wandb.login(key=wandb_key if wandb_key else None)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 72.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.2/212.2 kB 4.8 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.9/460.9 kB 12.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.5 MB/s eta 0:00:00


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jlakshya06 (jlakshya06-new-york-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Install dependencies (no need to open this)

In [5]:
!pip install -q mujoco==3.2.7 mujoco-mjx==3.2.7 brax==0.10.5 flax==0.10.2 orbax==0.1.9
!pip install black[jupyter] --quiet
import os
# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags
# Clean up incompatible jax versions
!pip uninstall -y jax jaxlib optax orbax-checkpoint

# Install specific compatible jax versions
!pip install optax==0.2.2 orbax-checkpoint==0.11.10
!pip install "jax[cuda12]==0.5.0" "jax-cuda12-plugin==0.5.0" "jax-triton==0.2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 25.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 93.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 21.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 68.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.9/998.9 kB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.2/424.2 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [6]:
#@title Check if MuJoCo installation was successful

from google.colab import files

import distutils.util
import os
import subprocess
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [ ]:
#@title Import packages for plotting and creating graphics
import time
import itertools
import numpy as np
from typing import Callable, NamedTuple, Optional, Union, List

# Graphics and plotting.
print('Installing mediapy:')
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [ ]:
#@title Import MuJoCo, MJX, and Brax


from datetime import datetime
import functools
from IPython.display import HTML
import jax
from jax import numpy as jp
import numpy as np
from typing import Any, Dict, Sequence, Tuple, Union

from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.envs.base import Env, PipelineEnv, State
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from brax.io import html, mjcf, model

from etils import epath
from flax import struct
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
from mujoco import mjx

In [ ]:
!git clone https://github.com/lakshyajain06/Pupper_Loco_Manipulation.git


# Config

## pupperv3-mjx repo config (no need to open this)

##Reward Configuration
Below are the reward coefficients that you will adjust to guide the policy toward maximizing the desired behaviors.

Most of the work for this lab will focus on **tuning these coefficients**!

Be sure to carefully review the lab instructions to understand the exact implementation of each reward term — this is crucial for deciding whether each coefficient should be positive or negative.

In [ ]:
reward_config = config_dict.ConfigDict()
    reward_config.rewards = config_dict.ConfigDict()
    reward_config.rewards.scales = config_dict.ConfigDict()

    # Track linear velocity
    # reward_config.rewards.scales.tracking_lin_vel = 20.0

    # Track the angular velocity along z-axis, i.e. yaw rate.
    # reward_config.rewards.scales.tracking_ang_vel = 10
    
    reward_config.rewards.tracking_foot_lin_pos = 10.0
    reward_config.rewards.scales.stand = 5.0

    # Track the given body orientation (desired world z axis in body frame)
    # Not working right nowkick
    reward_config.rewards.scales.tracking_orientation = 0

    # Below are regularization terms, we roughly divide the
    # terms to base state regularizations, joint
    # regularizations, and other behavior regularizations.
    # Penalize the base velocity in z direction, L2 penalty.
    reward_config.rewards.scales.lin_vel_z = -1.0

    # Penalize the base roll and pitch rate. L2 penalty.
    reward_config.rewards.scales.ang_vel_xy = -0.1

    # Penalize non-zero roll and pitch angles. L2 penalty.
    reward_config.rewards.scales.orientation = -0.1

    # L2 regularization of joint torques, sum(|tau|^2).
    reward_config.rewards.scales.torques = -0.0005

    # L2 regularization of joint accelerations sum(|qdd|^2)
    reward_config.rewards.scales.joint_acceleration = -0.001

    # L1 regularization of mechanical work, |v * tau|.
    reward_config.rewards.scales.mechanical_work = -0.01

    # Penalize the change in the action and encourage smooth
    # actions. L1 regularization |action - last_action|^2
    reward_config.rewards.scales.action_rate = -0.01

    # Encourage long swing steps. However, it does not
    # encourage high clearances.
    # reward_config.rewards.scales.feet_air_time = 1.5

    # Encourage joints at default position at zero command, L1 regularization
    # |q - q_default|.
    # reward_config.rewards.scales.stand_still = 1

    # Encourage zero joint velocity at zero command, L1 regularization
    # |q_dot|.
    # Activates when norm(command) < stand_still_command_threshold
    # Commands below this threshold are sampled with probability zero_command_probability
    # reward_config.rewards.scales.stand_still_joint_velocity = 1

    # Encourage zero abduction angle so legs don't spread so far out
    # L2 loss on ||abduction_motors - desired||^2
    reward_config.rewards.scales.abduction_angle = 0.1

    # Early termination penalty.
    reward_config.rewards.scales.termination = 0

    # Penalizing foot slipping on the ground.
    reward_config.rewards.scales.foot_slip = 0

    # Penalize knees hitting the ground
    reward_config.rewards.scales.knee_collision = -1.5

    # Penalize body hitting ground
    reward_config.rewards.scales.body_collision = -3.0

    # Tracking reward = exp(-error^2/sigma).
    reward_config.rewards.tracking_sigma = 0.25

# Modify robot model

This section modifies the mjx environment. You won't need to know what's going on here for this lab :-)

# Train Policy


In [ ]:
from Pupper_Loco_Manipulation.train import run_train

run_train(reward_config)

In [ ]:
# Save params to a model
model_path = os.path.join(output_folder, f'mjx_params_{train_datetime}')
model.save_params(model_path, params)

# Visualize Policy

For the Barkour Quadruped, the joystick commands can be set through `x_vel`, `y_vel`, and `ang_vel`. `x_vel` and `y_vel` define the linear forward and sideways velocities with respect to the quadruped torso. `ang_vel` defines the angular velocity of the torso in the z direction.

In [ ]:
inference_fn = make_inference_fn(params)
jit_inference_fn = jax.jit(inference_fn)
eval_env = envs.get_environment(env_name, **env_kwargs)

In [ ]:
# @markdown Commands **only used for Pupper Env**:
x_vel = 0.75  #@param {type: "number"}
y_vel = 0.  #@param {type: "number"}
ang_vel = 0.  #@param {type: "number"}

the_command = jp.array([x_vel, y_vel, ang_vel])

# initialize the state
rng = jax.random.PRNGKey(0)
state = jit_reset(rng)
state.info['command'] = the_command
pitch_rad = 0.
state.info['desired_world_z_in_body_frame'] = jp.array([jp.sin(pitch_rad),
                                                        0.0,
                                                        jp.cos(pitch_rad)])
rollout = [state.pipeline_state]

# grab a trajectory
n_steps = 300
render_every = 2
ctrls = []

for i in range(n_steps):
  act_rng, rng = jax.random.split(rng)

  ctrl, _ = jit_inference_fn(state.obs, act_rng)
  state = jit_step(state, ctrl)
  rollout.append(state.pipeline_state)
  ctrls.append(ctrl)

media.show_video(
    eval_env.render(rollout[::render_every], camera='tracking_cam'),
    fps=1.0 / eval_env.dt / render_every)

In [ ]:
import numpy as np
from pupperv3_mjx import plotting
plotting.plot_multi_series(data=np.array(ctrls), dt=0.02, display_axes=[0,1,2], title="Policy output")

In [ ]:
torques = jp.array([rollout[i].qfrc_actuator[6:] for i in range(len(rollout))]) # ignore world-body joint
plotting.plot_multi_series(data=torques, dt=0.02, display_axes=[0, 1, 2], title="Joint torques")

In [ ]:
joint_pos = jp.array([rollout[i].q[7:] for i in range(len(rollout))]) # ignore world-body joint
plotting.plot_multi_series(data=joint_pos, dt=0.02, display_axes=[0, 1, 2], title="Joint Positions")

In [ ]:
world_vel = jp.array([rollout[i].qvel[:3] for i in range(len(rollout))])
plotting.plot_multi_series(data=world_vel, dt=0.02, display_axes=[0, 1, 2], title="World Velocity")

# Export Policy for neural_controller

After running the following cells, open the files tab on the left and download policy.json.

In [ ]:
import json
from pupperv3_mjx import export
params_rtneural = export.convert_params(jax.block_until_ready(params),
                                        activation=CONFIG.policy.activation,
                                        action_scale=CONFIG.policy.action_scale,
                                        kp=CONFIG.training.position_control_kp,
                                        kd=CONFIG.training.dof_damping,
                                        default_pose=CONFIG.training.default_pose,
                                        joint_upper_limits=CONFIG.simulation.joint_upper_limits,
                                        joint_lower_limits=CONFIG.simulation.joint_lower_limits,
                                        use_imu=CONFIG.policy.use_imu,
                                        observation_history=CONFIG.policy.observation_history,
                                        final_activation="tanh",
                                        )

name = f"policy.json"
saved_policy_filename = os.path.join(output_folder, name)
with open(saved_policy_filename, "w") as f:
  json.dump(params_rtneural, f)

wandb.log_model(path=saved_policy_filename, name=name)
wandb.log_model(path=model_path, name=f"mjx_policy_network_{wandb.run.name}.pt")
wandb.finish()

After you are done training and have downloaded the policy, remember to shut down the runtime to save money!